In [2]:
import pandas as pd

df = pd.read_excel("C:/Users/DELL/Downloads/hotel_daily_booking_data_2024_2025.xlsx",skiprows=1)
df['date'] = pd.to_datetime(df['date'])

In [3]:
df.head()

,date,room_type,bookings,price_inr,occupancy_pct,event_flag,event_name,day_of_week,is_weekend
0,2024-01-01,Deluxe,57,16000,89.7,1,New Year,Monday,0
1,2024-01-01,Suite,14,30650,64.5,1,New Year,Monday,0
2,2024-01-01,Standard,83,11350,86.8,1,New Year,Monday,0
3,2024-01-02,Deluxe,51,12950,81.6,0,NaN,Tuesday,0
4,2024-01-02,Suite,13,24150,61.5,0,NaN,Tuesday,0


In [4]:
#room_counts = df.groupby(["room_type"])['room_type'].count()

In [5]:
#room_counts

In [6]:
print(df.shape)

(2193, 9)


In [7]:
df = df.sort_values(['room_type','date'])

In [8]:
df['month'] = df['date'].dt.month
#df['day'] = df['date'].dt.day
df['week'] = df['date'].dt.isocalendar().week.astype(int)

In [9]:
df = df.sort_values(['room_type', 'date'])

df['lag_1'] = df.groupby('room_type')['bookings'].shift(1)


df['lag_7'] = df.groupby('room_type')['bookings'].shift(7)


df['rolling_7'] =  df.groupby('room_type')['bookings'].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())

df['price_lag_1']      = df.groupby('room_type')['price_inr'].shift(1)
df['price_lag_7']      = df.groupby('room_type')['price_inr'].shift(7)
df['price_rolling_7']  = df.groupby('room_type')['price_inr'].transform(lambda x: x.shift(1).rolling(7).mean())


df = df.dropna(subset=['lag_1', 'lag_7', 'rolling_7', 'price_lag_1', 'price_lag_7', 'price_rolling_7'])


In [10]:
#df[(df['room_type'].isin(['Deluxe','Suite'])) & (df['date']<='2024-01-08')]
df[['lag_1', 'lag_7', 'rolling_7', 'price_lag_1', 'price_lag_7', 'price_rolling_7']].isna().sum()

lag_1              0
lag_7              0
rolling_7          0
price_lag_1        0
price_lag_7        0
price_rolling_7    0
dtype: int64

In [11]:
#df.head()

In [12]:
from sklearn.preprocessing import LabelEncoder
encoders = {}
for col in ['room_type','day_of_week']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

df = pd.get_dummies(df,columns=['event_name'],drop_first=True)

In [13]:
print(encoders['day_of_week'].classes_)
print(encoders['room_type'].classes_)
print(encoders)

['Friday' 'Monday' 'Saturday' 'Sunday' 'Thursday' 'Tuesday' 'Wednesday']
['Deluxe' 'Standard' 'Suite']
{'room_type': LabelEncoder(), 'day_of_week': LabelEncoder()}


In [14]:
for col, le in encoders.items():
    mapping = dict(zip(le.transform(le.classes_), le.classes_))
    print(f"{col}: {mapping}")


room_type: {np.int64(0): 'Deluxe', np.int64(1): 'Standard', np.int64(2): 'Suite'}
day_of_week: {np.int64(0): 'Friday', np.int64(1): 'Monday', np.int64(2): 'Saturday', np.int64(3): 'Sunday', np.int64(4): 'Thursday', np.int64(5): 'Tuesday', np.int64(6): 'Wednesday'}


In [15]:
##rows = []
##for col, le in encoders.items():
   ## for encoded, original in zip(le.transform(le.classes_), le.classes_):
        ##rows.append({'column': col, 'encoded': encoded, 'original': original})

##pd.DataFrame(rows).to_excel("C:/Users/DELL/Downloads/label_encoding_mapping.xlsx", index=False)
#print("Saved ✅")


# ── Inverse transform later ──────────────────────────────────────────────────
print(encoders['room_type'].inverse_transform([0, 1, 2]))    # → ['Deluxe', 'Standard', 'Suite']
print(encoders['day_of_week'].inverse_transform([0, 1, 2,3,4,5,6]))  # → ['Friday', 'Monday', 'Saturday']


['Deluxe' 'Standard' 'Suite']
['Friday' 'Monday' 'Saturday' 'Sunday' 'Thursday' 'Tuesday' 'Wednesday']


In [13]:
#future['room_type'] = encoders['room_type'].inverse_transform(future['room_type'])
#encoders['room_type'].inverse_transform(future['room_type'])


In [16]:
print(df.columns)

Index(['date', 'room_type', 'bookings', 'price_inr', 'occupancy_pct',
       'event_flag', 'day_of_week', 'is_weekend', 'month', 'week', 'lag_1',
       'lag_7', 'rolling_7', 'price_lag_1', 'price_lag_7', 'price_rolling_7',
       'event_name_Diwali', 'event_name_New Year', 'event_name_Tamil New Year',
       'event_name_Valentine's Day'],
      dtype='str')


In [17]:
df = df.sort_values(['room_type', 'date'])

In [18]:
cutoff_date = (df['date'].max() - pd.Timedelta(days=30))
train_df = df[df['date'] <= cutoff_date]
val_df = df[df['date'] > cutoff_date]

print(cutoff_date)
print(train_df['date'].min(), train_df['date'].max())
print(val_df['date'].min(), val_df['date'].max())

2025-12-01 00:00:00
2024-01-08 00:00:00 2025-12-01 00:00:00
2025-12-02 00:00:00 2025-12-31 00:00:00


In [19]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [20]:
exog_features = ['day_of_week','is_weekend', 'month', 'week',
                   'price_lag_1', 'price_lag_7', 'price_rolling_7',
                      'event_name_Diwali', 'event_name_New Year',
                        'event_name_Tamil New Year', "event_name_Valentine's Day"
]

In [22]:
for col in exog_features:
    train_df[col] = train_df[col].astype(float)
    val_df[col]   = val_df[col].astype(float)

In [23]:
for col in exog_features:
    print(col, train_df[col].dtype, train_df[col].dtype == object)

day_of_week float64 False
is_weekend float64 False
month float64 False
week float64 False
price_lag_1 float64 False
price_lag_7 float64 False
price_rolling_7 float64 False
event_name_Diwali float64 False
event_name_New Year float64 False
event_name_Tamil New Year float64 False
event_name_Valentine's Day float64 False


In [39]:
#print(arimax_results['Deluxe']['preds'].index)

In [48]:
# ── Rename columns with spaces/special chars ──────────────────────────────────
rename_map = {
    'event_name_New Year':        'event_name_New_Year',
    'event_name_Tamil New Year':  'event_name_Tamil_New_Year',
    "event_name_Valentine's Day": 'event_name_Valentines_Day'
}
train_df = train_df.rename(columns=rename_map)
val_df   = val_df.rename(columns=rename_map)
df       = df.rename(columns=rename_map)

